# Empirical Performance Evaluation of Node.js, Bun and Deno

## Experiment Overview

This notebook documents the design and implementation of an empirical performance comparison of Node.js, Bun and Deno.

Unlike studies that rely on an existing dataset, this project will generate a primary experimental dataset through controlled benchmark executions. The dataset will contain measurements relating to HTTP throughput and latency, cold-start time, file I/O performance and memory usage.

The generated data will be validated, preprocessed, analysed and visualised using R.

## Study Variables

### Independent Variable

The independent variable is the server-side JavaScript runtime used to execute each workload.

The runtime levels are:

- Node.js
- Bun
- Deno

### Dependent Variables

The dependent variables are:

1. HTTP throughput, measured in requests per second.
2. HTTP latency, measured in milliseconds.
3. Cold-start time, measured in milliseconds.
4. File I/O performance, measured using operation time and throughput.
5. Memory usage, measured primarily using resident set size.

## Benchmark Categories

### HTTP Throughput and Latency

Equivalent HTTP servers will be implemented using Node.js, Bun and Deno. A separate load-generation tool will send controlled requests to each server.

The experiment will record:

- Requests per second.
- Mean latency.
- Median latency.
- 95th-percentile latency.
- 99th-percentile latency.
- Failed requests.
- Timeouts.

### Cold-Start Time

Each runtime will be launched as a new process. Cold-start time will initially be defined as the time between process creation and the point at which the HTTP server becomes ready to accept requests.

### File I/O Performance

The runtimes will perform equivalent file-reading and file-writing tasks using identical test files.

Possible workloads include:

- Reading a large file.
- Writing a large file.
- Reading multiple small files.
- Writing multiple small files.

### Memory Usage

Memory usage will be measured externally at operating-system level where possible.

Possible measurements include:

- Baseline resident memory.
- Average resident memory.
- Peak resident memory.
- Memory increase under workload.

## Controlled Variables

To support a fair comparison, the following conditions will remain consistent:

| Controlled variable | Control method |
|---|---|
| Hardware | All experiments will run on the same computer |
| Operating system | The same operating system and configuration will be used |
| Runtime architecture | All runtimes will use the same system architecture |
| HTTP response | Each server will return the same status, headers and payload |
| Load generator | The same tool and version will test all runtimes |
| Concurrency level | Identical concurrency settings will be used |
| Test duration | Identical measurement durations will be used |
| File fixtures | All runtimes will use identical files |
| Repetitions | Each runtime will receive the same number of measured runs |
| Warm-up policy | The same benchmark-specific warm-up procedure will be applied |
| Network | HTTP benchmarks will run through localhost |
| Background activity | Unnecessary applications and services will be minimised |
| Error handling | Failures and timeouts will be recorded rather than silently removed |

In [1]:
R.version.string

[1] "R version 4.6.0 (2026-04-24 ucrt)"

In [ ]:
# record the R version, operating system and attached packages available when the notebook was created.
sessionInfo()

R version 4.6.0 (2026-04-24 ucrt)
Platform: x86_64-w64-mingw32/x64
Running under: Windows 10 x64 (build 18363)

Matrix products: default
  LAPACK version 3.12.1

locale:
[1] LC_COLLATE=English_United Kingdom.utf8 
[2] LC_CTYPE=English_United Kingdom.utf8   
[3] LC_MONETARY=English_United Kingdom.utf8
[4] LC_NUMERIC=C                           
[5] LC_TIME=English_United Kingdom.utf8    

time zone: Europe/London
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] digest_0.6.39     IRdisplay_1.1     base64enc_0.1-6   fastmap_1.2.0    
 [5] glue_1.8.1        htmltools_0.5.9   repr_1.1.7        lifecycle_1.0.5  
 [9] cli_3.6.6         vctrs_0.7.3       textshaping_1.0.5 pbdZMQ_0.3-14    
[13] systemfonts_1.3.2 compiler_4.6.0    tools_4.6.0       ragg_1.5.2       
[17] evaluate_1.0.5    pillar_1.11.1     rlang_1.2.0       jsonlite_2.0.0   
[21] crayon_1.5.3      IRkernel_

In [3]:
getwd()

[1] "d:/FOLO_PROJECTS/MASTERS/javascript-runtime-performance-study/notebooks"

## Data-Generation Process

The experimental dataset will be generated using the following process:

1. Define a benchmark workload.
2. Implement equivalent versions for Node.js, Bun and Deno.
3. Validate that each implementation produces the same functional result.
4. Conduct a small pilot experiment.
5. inspect the pilot measurements and identify design problems.
6. Revise and freeze the benchmark configuration.
7. Perform repeated final benchmark executions.
8. Store the original measurements as immutable raw data.
9. Validate and preprocess the data using R.
10. Conduct exploratory and statistical analysis.
11. Export dissertation-ready figures and tables.

Pilot data will be stored separately from the final experimental dataset.

## Initial Experimental Decisions

| Decision | Current position | Status |
|---|---|---|
| Analysis language | R | Confirmed |
| Documentation environment | Local Jupyter Notebook | Confirmed |
| Compared runtimes | Node.js, Bun and Deno | Confirmed |
| HTTP measures | Throughput and latency | Confirmed |
| Cold-start definition | Process launch to HTTP server readiness | Provisional |
| File I/O workloads | Large-file and small-file read/write operations | Provisional |
| Main memory measure | Resident set size | Provisional |
| Number of repetitions | To be established through pilot testing | Undecided |
| Warm-up procedure | Benchmark-specific | Undecided |
| HTTP load-testing tool | To be selected | Undecided |

## Experimental Environment

All benchmark experiments will be conducted on the same physical computer. The hardware, operating system, runtime versions and analysis environment are recorded to support reproducibility and interpretation of the results.

The environment information below was captured before benchmark development and will be checked again before final data collection.

In [4]:
environment_summary <- data.frame(
  property = c(
    "Recorded at",
    "Operating system",
    "Operating system release",
    "System architecture",
    "R version"
  ),
  value = c(
    format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z"),
    Sys.info()[["sysname"]],
    Sys.info()[["release"]],
    Sys.info()[["machine"]],
    R.version.string
  )
)

environment_summary

property,value
<chr>,<chr>
Recorded at,2026-07-12 20:40:26 BST
Operating system,Windows
Operating system release,10 x64
System architecture,x86-64
R version,R version 4.6.0 (2026-04-24 ucrt)


In [6]:
# Record Processor,memory, and windows details
run_powershell <- function(command) {
  result <- tryCatch(
    system2(
      "powershell",
      args = c("-NoProfile", "-Command", shQuote(command)),
      stdout = TRUE,
      stderr = TRUE
    ),
    error = function(e) {
      paste("Unable to retrieve information:", conditionMessage(e))
    }
  )

  paste(result, collapse = "\n")
}

In [7]:
cpu_information <- run_powershell(
  "Get-CimInstance Win32_Processor |
   Select-Object Name, NumberOfCores, NumberOfLogicalProcessors |
   Format-List"
)

memory_information <- run_powershell(
  "Get-CimInstance Win32_ComputerSystem |
   Select-Object @{Name='TotalPhysicalMemoryGB';
   Expression={[math]::Round($_.TotalPhysicalMemory / 1GB, 2)}} |
   Format-List"
)

windows_information <- run_powershell(
  "Get-CimInstance Win32_OperatingSystem |
   Select-Object Caption, Version, OSArchitecture |
   Format-List"
)

cat("PROCESSOR\n")
cat(cpu_information)

cat("\n\nMEMORY\n")
cat(memory_information)

cat("\n\nOPERATING SYSTEM\n")
cat(windows_information)

PROCESSOR


Name                      : Intel(R) Core(TM) i5-6200U CPU @ 2.30GHz
NumberOfCores             : 2
NumberOfLogicalProcessors : 4




MEMORY


TotalPhysicalMemoryGB : 15.84




OPERATING SYSTEM


Caption        : Microsoft Windows 10 Enterprise
Version        : 10.0.18363
OSArchitecture : 64-bit




In [9]:
get_command_version <- function(command, args = "--version") {
  result <- tryCatch(
    suppressWarnings(
      system2(
        command,
        args,
        stdout = TRUE,
        stderr = TRUE
      )
    ),
    error = function(e) character(0)
  )

  if (
    length(result) == 0 ||
    !is.null(attr(result, "status"))
  ) {
    return("Not detected")
  }

  paste(result, collapse = " ")
}



In [10]:
software_versions <- data.frame(
  software = c(
    "Node.js",
    "Bun",
    "Deno",
    "R",
    "Jupyter",
    "Git"
  ),
  version = c(
    get_command_version("node"),
    get_command_version("bun"),
    get_command_version("deno"),
    R.version.string,
    get_command_version("jupyter", "--version"),
    get_command_version("git")
  )
)

software_versions

software,version
<chr>,<chr>
Node.js,v20.12.2
Bun,Not detected
Deno,Not detected
R,R version 4.6.0 (2026-04-24 ucrt)
Jupyter,Selected Jupyter core packages... IPython : 9.15.0 ipykernel : 7.3.0 ipywidgets : not installed jupyter_client : 8.9.1 jupyter_core : 5.9.1 jupyter_server : 2.20.0 jupyterlab : 4.6.1 nbclient : 0.11.0 nbconvert : 7.17.1 nbformat : 5.10.4 notebook : not installed qtconsole : not installed traitlets : 5.15.1
Git,git version 2.41.0.windows.3


### Environment Recording Notes

The environment information above represents the initial development environment. All three runtimes must be installed and accessible through the system PATH before benchmark implementation begins.

Runtime and package versions will be recorded again immediately before final data collection. Final experiments will be conducted on the same machine and operating system configuration.

In [1]:
get_command_output <- function(command, args = character()) {
  tryCatch(
    {
      result <- system2(
        command,
        args = args,
        stdout = TRUE,
        stderr = TRUE
      )

      paste(result, collapse = " ")
    },
    error = function(e) {
      "Not detected"
    }
  )
}

In [2]:
runtime_versions <- data.frame(
  runtime = c("Node.js", "Bun", "Deno"),
  version = c(
    get_command_output("node", "--version"),
    get_command_output("bun", "--version"),
    get_command_output("deno", "--version")
  )
)

runtime_versions

runtime,version
<chr>,<chr>
Node.js,v20.12.2
Bun,1.3.14
Deno,"deno 2.9.2 (stable, release, x86_64-pc-windows-msvc) v8 14.9.207.2-rusty typescript 6.0.3"


In [3]:
get_windows_executable_path <- function(command) {
  tryCatch(
    {
      result <- system2(
        "where.exe",
        args = command,
        stdout = TRUE,
        stderr = TRUE
      )

      paste(result, collapse = "; ")
    },
    error = function(e) {
      "Not detected"
    }
  )
}

In [4]:
runtime_paths <- data.frame(
  runtime = c("Node.js", "Bun", "Deno"),
  executable_path = c(
    get_windows_executable_path("node"),
    get_windows_executable_path("bun"),
    get_windows_executable_path("deno")
  )
)

runtime_paths

runtime,executable_path
<chr>,<chr>
Node.js,C:\Program Files\nodejs\node.exe
Bun,D:\Bun\bin\bun.exe
Deno,D:\Deno\bin\deno.exe
